# Inference Suboptimality on a Kaggle GPU

Before running, in the right-hand panel under **Session options**:
- **Accelerator:** `GPU T4` (the code uses a single GPU; the P100 is not supported by recent PyTorch builds)
- **Internet:** On (needed for `git clone` and the CIFAR download)

For long runs use **Save Version → Save & Run All**: it runs in the background (up to 12 h) and
keeps everything in `/kaggle/working` as the version's output.

In [ ]:
# --- Configuration -----------------------------------------------------------
REPO = "https://github.com/ugwnate-hash/inference-suboptimality.git"
BRANCH = "my-work"

DATASET = "mnist"              # mnist | fashion | cifar
MODEL_FLAGS = ""               # e.g. "--has-flow", "--wide-encoder", "--has-flow --hamiltonian-flow"
TRAIN_FLAGS = "--lr-schedule --warmup --early-stopping"
EPOCHS = 3280
DISPLAY_EPOCH = 10             # evaluate + save a checkpoint every N epochs

## 1. Get the code

In [ ]:
import os
os.chdir("/kaggle/working")
if not os.path.isdir("inference-suboptimality"):
    !git clone -b {BRANCH} {REPO}
else:
    !git -C inference-suboptimality pull
os.chdir("/kaggle/working/inference-suboptimality")
!git log --oneline -1

## 2. Check the GPU and install dependencies

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print("GPU:", torch.cuda.get_device_name(0))
    if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        print("WARNING: this PyTorch build does not support this GPU; switch the accelerator to GPU T4")
else:
    print("WARNING: no GPU; enable it under Session options -> Accelerator")

!pip install -q -r requirements.txt

## 3. Train

Checkpoints go to `checkpoints/<dataset>/<warmup>/<model>/<epoch>_model.pth` and the log to `logs/`.
To resume an interrupted run, add `--load-path <ckpt> --load-epoch <epoch>` to `TRAIN_FLAGS`.
`TQDM_DISABLE=1` turns off the per-iteration progress bars, which would otherwise flood the log.

In [ ]:
!mkdir -p logs
!TQDM_DISABLE=1 python -u run.py --train --dataset {DATASET} --epochs {EPOCHS} --display-epoch {DISPLAY_EPOCH} {MODEL_FLAGS} {TRAIN_FLAGS} 2>&1 | tee -a logs/train_{DATASET}.log

In [ ]:
# Pick the most recent checkpoint (or set CKPT by hand)
import glob
checkpoints = glob.glob(f"checkpoints/{DATASET}/**/*_model.pth", recursive=True)
CKPT = max(checkpoints, key=os.path.getmtime)
print("Evaluating", CKPT)

## 4. Evaluate

These settings are moderate so each cell finishes in reasonable time. The paper's settings are
`--n-iwae 5000` and `--n-ais-dist 10000 --n-ais-iwae 100`; increase gradually.

In [ ]:
# ELBO (100 MC samples) and IWAE bound with the amortized encoder
!TQDM_DISABLE=1 python -u run.py --eval-iwae --dataset {DATASET} --eval-path {CKPT} {MODEL_FLAGS} --n-iwae 1000 2>&1 | tee logs/eval_iwae_{DATASET}.log

In [ ]:
# AIS estimate of log p(x) (close to the true marginal likelihood)
!TQDM_DISABLE=1 python -u run.py --eval-ais --dataset {DATASET} --eval-path {CKPT} {MODEL_FLAGS} --n-ais-dist 1000 --n-ais-iwae 16 2>&1 | tee logs/eval_ais_{DATASET}.log

In [ ]:
# Locally optimized Gaussian per datapoint (removes the amortization gap)
# --hamiltonian-flow is not accepted by this script
!TQDM_DISABLE=1 python -u local_ffg.py --dataset {DATASET} --eval-path {CKPT} {MODEL_FLAGS} 2>&1 | tee logs/local_ffg_{DATASET}.log

In [ ]:
# Locally optimized flow per datapoint (slow: one datapoint at a time)
# !TQDM_DISABLE=1 python -u local_flow.py --dataset {DATASET} --eval-path {CKPT} {MODEL_FLAGS} 2>&1 | tee logs/local_flow_{DATASET}.log

In [ ]:
# Bidirectional Monte Carlo on simulated data: checks how tight AIS is
!TQDM_DISABLE=1 python -u bdmc.py --dataset {DATASET} --eval-path {CKPT} {MODEL_FLAGS} --n-ais-dist 1000 --n-ais-iwae 16 --n-batch 1 2>&1 | tee logs/bdmc_{DATASET}.log

## 5. Package results

Everything under `/kaggle/working` is kept as this version's output. The archive below is a single
file to download, or to attach to another notebook via **Output → New Dataset**.

In [ ]:
!tar -czf /kaggle/working/results_{DATASET}.tar.gz checkpoints logs
!ls -lh /kaggle/working/results_{DATASET}.tar.gz